# Process Mining Group 5 notebook

## Setup

In [1]:
import pm4py
import pandas as pd
from pathlib import Path

## Load datasets

In [2]:
domesticDeclarations = pm4py.read_xes("Datasets/domesticDeclarations.xes")
internationalDeclarations = pm4py.read_xes("Datasets/internationalDeclarations.xes")
permitLog = pm4py.read_xes("Datasets/permitLog.xes")
prepaidTravelCost = pm4py.read_xes("Datasets/prepaidTravelCost.xes")
requestForPayment = pm4py.read_xes("Datasets/requestForPayment.xes")

df_domesticDeclarations = pm4py.convert_to_dataframe(domesticDeclarations)
df_internationalDeclarations = pm4py.convert_to_dataframe(internationalDeclarations)
df_permitLog = pm4py.convert_to_dataframe(permitLog)
df_prepaidTravelCost = pm4py.convert_to_dataframe(prepaidTravelCost)
df_requestForPayment = pm4py.convert_to_dataframe(requestForPayment)

c:\Users\tomwo\Documents\GitHub\process-mining-group5\venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
c:\Users\tomwo\Documents\GitHub\process-mining-group5\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 6886/6886 [00:00<00:00, 12095.49it/s]


#### Export datasets (not required)

In [3]:
Path("Output").mkdir(exist_ok=True)

df_domesticDeclarations.to_csv("Output/domesticDeclarations.csv", index=False)
df_internationalDeclarations.to_csv("Output/internationalDeclarations.csv", index=False)
df_permitLog.to_csv("Output/permitLog.csv", index=False)
df_prepaidTravelCost.to_csv("Output/prepaidTravelCost.csv", index=False)
df_requestForPayment.to_csv("Output/requestForPayment.csv", index=False)

## Combine datasets

In [4]:
rfp_matched = df_requestForPayment.merge(
    df_prepaidTravelCost[["case:Rfp_id", "id"]],
    on="case:Rfp_id",
    how="inner",
    suffixes=("_rfp", "_ptc")
)
rfp_matched["id"] = rfp_matched["id_ptc"]
rfp_matched = rfp_matched.drop(columns=["id_rfp", "id_ptc"])

df_final = pd.concat(
    [
        df_domesticDeclarations,
        df_internationalDeclarations,
        df_permitLog,
        df_prepaidTravelCost,
        rfp_matched
    ],
    ignore_index=True
)

## Pre-process datasets

In [5]:
df_final["time:timestamp"] = pd.to_datetime(
    df_final["time:timestamp"],
    errors="coerce",
    utc=True
)

df_final = df_final[
    (df_final["time:timestamp"].dt.year >= 2017) &
    (df_final["time:timestamp"].dt.year <= 2019)
]

df_final = df_final.drop_duplicates(
    subset=["id", "concept:name", "time:timestamp"]
)

df_final = df_final.sort_values(
    [
        "case:concept:name",
        "time:timestamp",
        "id",
        "concept:name"
    ],
    kind="mergesort"
).reset_index(drop=True)

#### Export final dataset (not required)

In [6]:
df_final.to_csv("Output/df_final.csv", index=False)

event_log = pm4py.convert_to_event_log(df_final)

pm4py.write_xes(event_log, "Output/df_final.xes")

exporting log, completed traces :: 100%|██████████| 19803/19803 [00:08<00:00, 2264.68it/s]


## Create petri net

In [ ]:
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(
    df_final,
    case_id_key="id",
    activity_key="concept:name",
    timestamp_key="time:timestamp",
    noise_threshold=0.0,
    multi_processing=False
)

pm4py.save_vis_petri_net(
    net,
    initial_marking,
    final_marking,
    "petri_net.png"
)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH